# V16 — Neo Normal thích ứng theo video (lộ trình mới, 14/11/2026)

**Đây là V16 mới.** Thử bước 1 trước: đóng băng toàn bộ checkpoint V0, chỉ thay
vector Normal cuối của A-Branch bằng hỗn hợp với feature bình thường trong cùng video.
Không huấn luyện, không thay prompt lớp bất thường, không thay LGT, không trộn hai nhánh.
Neo Normal gốc của VadCLIP đã có visual prompting theo block; V16 giữ neo gốc đó
làm fallback và thêm một neo nhìn lấy từ toàn video. Không coi neo gốc là text cố định thuần túy.

## Bốn input bắt buộc — đúng các gói đã chạy V0/V7

| Input | Nội dung |
|---|---|
| UCF test | `ucfclipfeatures_test290.zip` (290 video, crop 5) hoặc cây feature đã giải nén |
| Checkpoint UCF | `vadclip_ucf_checkpoint_upload.zip` chứa `model_ucf.pth`, hoặc file `.pth` |
| **VadCLIP XD-Violence Test 800** | `vadclip_xd_test800_kaggle_v2.zip` và manifest đã dùng ở V0; tên Dataset giữ nguyên |
| Checkpoint XD | `vadclip_xd_checkpoint.zip` chứa checkpoint V0, hoặc file `.pth` |

**Không cần output V15, train features, I3D, video MP4 hoặc audio.** Dùng Add Input
gắn các Dataset cũ; không cần đổi tên Dataset. Nếu gắn nhiều bản cùng loại, điền
đường dẫn cụ thể ở cell cấu hình. Không tắt kiểm hash để vượt lỗi input.

Kaggle: GPU T4, Internet ON để lấy mã VadCLIP commit đã khóa và CLIP ViT-B/16.
`CLIP_WEIGHTS` tùy chọn là đường dẫn `ViT-B-16.pt` OpenAI chính thức; không phải checkpoint VadCLIP.

## Cách chạy

1. Import notebook; để `RUN_MODE = "smoke"`; Run All.
2. Kiểm cả `ucf` và `xd` báo `ok`. Smoke đi qua input/hash, checkpoint thật và tính neo.
3. Đổi **duy nhất** `RUN_MODE = "full"`; Run All. Giữ cấu hình trước khi xem kết quả.
4. Tải `/kaggle/working/vadclip_v16/run_*/vadclip_v16_results.zip`.

Mỗi bộ chạy V0, đối chứng `identity` với beta=0, rồi V16. Full buộc identity khớp
bitwise cả hai nhánh và C-V16 khớp C-V0; dừng nếu V0 không khớp mốc tái lập.

## Công thức đã khóa cho phép thử đầu

Chỉ chọn snippet có sigmoid(C) ≤ 0,20 và 1-softmax(A)[Normal] ≤ 0,20,
đồng thời các hàng xóm thực liền kề cũng đạt. Lấy tối đa 20% snippet điểm thấp nhất,
cần ít nhất 8 snippet và cosine coherence ≥ 0,50. Tất cả lựa chọn chỉ dùng prediction,
không nhận nhãn video/frame test. Các ngưỡng là giả định thăm dò, chưa hiệu chuẩn.

Gom feature thị giác **sau adapter, trong đúng không gian dùng để tính A**, trọng số
(1-pC)(1-pA), rồi chuẩn hóa. Không thêm projection chưa được học. Beta tối đa 0,10,
giảm theo độ tin cậy, lượng bằng chứng và coherence. Không đủ điều kiện thì giữ nguyên.
Chỉ thay logit Normal cuối, giữ temperature gốc và công thức 1-softmax(A)[Normal].
Toàn bộ inference dùng no_grad/detach. Các block cùng một video chia sẻ neo bổ sung;
không trộn neo giữa video; loại padding trước khi chọn và gom feature.

Đây là phép suy luận **offline** dùng toàn video, không phải phát hiện nhân quả trực tuyến.

## Đọc kết quả

- `comparison.csv`: A-V16 so A-V0 và C-V16 so C-V0, AUC/AP hai bộ, Ano-AUC UCF.
- UCF lấy C làm mốc chính nên V16 này **phải giữ nguyên AUC chính UCF**. Việc A tăng
  chưa chứng minh hệ thống vượt mốc chính; không tự đổi sang nhánh tốt hơn sau khi xem test.
- `anchor_diagnostics.json`: beta, số snippet, chỉ số được chọn và lý do fallback từng video.
- `predictions.npz`: điểm hai nhánh, ID, offset và GT để replay; `run.json` lưu hash/metric.
- `false_alarm_diagnostics.csv`: FPR tại recall yêu cầu 50%/80%, kèm recall thực do ties.
  Ngưỡng này dùng nhãn test để phân tích hồi cứu, **không dùng triển khai hoặc chọn cấu hình**.
- CI bootstrap 500 lần lấy mẫu theo video, không coi các frame độc lập; không phản ánh nhiều seed train.
- Không có temporal mAP được kiểm chứng trong phép thử này; không tuyên bố bảo toàn fine-grained.

## Thời gian và giới hạn

Không có lượt train. Mỗi bộ chạy ba lượt inference và kiểm hash input.
Smoke in ước lượng full từ thời gian đo thật; độ dài video khác nhau làm ước lượng có sai số.
Dự trù ban đầu 10–30 phút trên T4 với input sẵn; đây chưa phải thời gian V16 đo trên Kaggle,
không gồm tải CLIP/mạng chậm. Nếu smoke lỗi, gửi ZIP/log trước khi chạy full.

## Các bước kế tiếp

V17: điều biến residual C→A, đối chứng riêng và kết hợp với V16.
V18: cổng biến thiên ngữ nghĩa trực chuẩn, chỉ điều biến cập nhật thời gian.
V19: hard-negative prompts với loss margin và đối chứng giữ nguyên công thức điểm.
Ba bước này **chưa được triển khai/huấn luyện trong notebook V16**; không ghép trước khi đọc kết quả bước 1.


In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN_MODE = "smoke"
DATASETS = ("ucf", "xd")
UCF_DATA_ROOT = None
UCF_CHECKPOINT = None
XD_DATA_ROOT = None
XD_FEATURE_ZIP = None
XD_CHECKPOINT = None
CLIP_WEIGHTS = None

# Fixed exploratory configuration: do not tune on official test results.
ANCHOR_CONFIG = {
    "beta_max": 0.10, "normal_c_max": 0.20, "normal_a_max": 0.20,
    "bottom_fraction": 0.20, "min_snippets": 8,
    "min_coherence": 0.50, "neighbor_radius": 1,
}
BOOTSTRAP_REPLICATES = 500
SEED = 42


In [ ]:
ANCHOR_SOURCE = '"""Frozen, offline per-video Normal anchor. No labels enter this module.\n\nNative inputs: normalized visual [blocks, width, D], normalized text [blocks, D, C],\nbinary logits [blocks, width, 1], alignment logits [blocks, width, C].\nAll blocks MUST belong to one video. Padding is excluded by lengths.\nOnly the Normal logit (column 0) is changed; native C and other A logits are retained.\n"""\n\nimport ast\nimport hashlib\nimport math\n\nimport torch\nimport torch.nn.functional as F\n\nDEFAULT_CONFIG = {\n    "beta_max": 0.10,\n    "normal_c_max": 0.20,\n    "normal_a_max": 0.20,\n    "bottom_fraction": 0.20,\n    "min_snippets": 8,\n    "min_coherence": 0.50,\n    "neighbor_radius": 1,\n}\n\n\ndef validate_config(cfg):\n    if set(cfg) != set(DEFAULT_CONFIG):\n        raise ValueError("Unexpected/missing anchor configuration keys")\n    for key in ("beta_max", "normal_c_max", "normal_a_max", "bottom_fraction", "min_coherence"):\n        if not isinstance(cfg[key], (int, float)) or not math.isfinite(cfg[key]):\n            raise ValueError(f"Invalid finite number: {key}")\n    if not (0 <= cfg["beta_max"] <= 1 and 0 < cfg["bottom_fraction"] <= 1):\n        raise ValueError("Invalid beta or bottom fraction")\n    if not all(0 < cfg[k] < 1 for k in ("normal_c_max", "normal_a_max", "min_coherence")):\n        raise ValueError("Confidence/coherence cutoffs must be in (0, 1)")\n    if type(cfg["min_snippets"]) is not int or cfg["min_snippets"] < 1:\n        raise ValueError("min_snippets must be a positive integer")\n    if type(cfg["neighbor_radius"]) is not int or cfg["neighbor_radius"] < 0:\n        raise ValueError("neighbor_radius must be a nonnegative integer")\n\n\ndef instrument_source(source):\n    """Capture the exact operands of native A matmul; fail closed on source drift.\n\nNo reimplementation of LGT, visual prompting, text encoder or logits. Parsing\nis restricted to the pinned author\'s source, never a checkpoint/data payload.\n"""\n    tree = ast.parse(source)\n    classes = [n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == "CLIPVAD"]\n    if len(classes) != 1:\n        raise ValueError("Expected one upstream CLIPVAD class")\n    methods = [n for n in classes[0].body if isinstance(n, ast.FunctionDef) and n.name == "forward"]\n    if len(methods) != 1:\n        raise ValueError("Expected one upstream forward")\n    method = methods[0]\n    returns = [n for n in method.body if isinstance(n, ast.Return)]\n    if len(returns) != 1 or not isinstance(returns[0].value, ast.Tuple):\n        raise ValueError("Native forward return structure changed")\n    result = returns[0]\n    if len(result.value.elts) != 3 or not isinstance(result.value.elts[2], ast.Name):\n        raise ValueError("Expected native (text, C logits, A logits)")\n    target = result.value.elts[2].id\n    assignments = [n for n in method.body if isinstance(n, ast.Assign)\n                   and any(isinstance(t, ast.Name) and t.id == target for t in n.targets)]\n    if len(assignments) != 1:\n        raise ValueError("Cannot identify native alignment computation uniquely")\n    expr = assignments[0].value\n    if not (isinstance(expr, ast.BinOp) and isinstance(expr.op, ast.Div)\n            and isinstance(expr.left, ast.BinOp) and isinstance(expr.left.op, ast.MatMult)\n            and isinstance(expr.right, ast.Constant)\n            and isinstance(expr.right.value, (int, float)) and expr.right.value > 0):\n        raise ValueError("Expected native visual @ text / positive literal temperature")\n    if not isinstance(expr.left.left, ast.Name):\n        raise ValueError("Unexpected visual operand; inspect pinned source")\n    # Permit native tensor dtype conversion, but no arbitrary function calls.\n    right = expr.left.right\n    if not (isinstance(right, ast.Name) or (\n        isinstance(right, ast.Call) and isinstance(right.func, ast.Attribute)\n        and isinstance(right.func.value, ast.Name) and right.func.attr == "type"\n    )):\n        raise ValueError("Unexpected text operand; inspect pinned source")\n    capture = ast.parse(\n        "self._v16_capture = ("\n        f"{ast.unparse(expr.left.left)}, {ast.unparse(right)}, {expr.right.value!r})"\n    ).body[0]\n    method.body.insert(method.body.index(result), capture)\n    ast.fix_missing_locations(tree)\n    return ast.unparse(tree) + "\\n"\n\n\ndef patch_model_file(path):\n    source = path.read_text(encoding="utf-8")\n    patched = instrument_source(source)\n    path.write_text(patched, encoding="utf-8")\n    return {"original_model_sha256": hashlib.sha256(source.encode()).hexdigest(),\n            "instrumented_model_sha256": hashlib.sha256(patched.encode()).hexdigest()}\n\n\n@torch.no_grad()\ndef adapt_normal(visual, text, logits_c, logits_a, lengths, cfg):\n    validate_config(cfg)\n    visual, text = visual.detach(), text.detach()\n    logits_c, logits_a = logits_c.detach(), logits_a.detach()\n    if visual.ndim != 3 or text.ndim != 3 or logits_a.ndim != 3:\n        raise ValueError("Expected block-major 3D tensors")\n    blocks, width, dim = visual.shape\n    classes = logits_a.shape[-1]\n    if (text.shape != (blocks, dim, classes) or logits_a.shape[:2] != (blocks, width)\n            or logits_c.shape != (blocks, width, 1) or classes < 2):\n        raise ValueError("Native embedding/logit shapes do not match")\n    sizes = torch.as_tensor(lengths, device=visual.device).reshape(-1)\n    if len(sizes) != blocks or (sizes < 0).any() or (sizes > width).any():\n        raise ValueError("Invalid native block lengths")\n    valid = torch.arange(width, device=visual.device)[None, :] < sizes[:, None]\n    if not valid.any() or any(not x.isfinite().all() for x in (visual, text, logits_c, logits_a)):\n        raise ValueError("Empty video or nonfinite native tensors")\n    v = visual[valid].float()\n    pc = logits_c[..., 0][valid].float().sigmoid()\n    pa = 1 - logits_a[valid].float().softmax(-1)[:, 0]\n    confidence = torch.maximum(pc, pa)\n    eligible = (pc <= cfg["normal_c_max"]) & (pa <= cfg["normal_a_max"])\n    # Require stable low scores in real adjacent snippets, including across blocks.\n    stable = eligible.clone()\n    for shift in range(1, cfg["neighbor_radius"] + 1):\n        stable[shift:] &= eligible[:-shift]\n        stable[:-shift] &= eligible[shift:]\n    indices = stable.nonzero().flatten()\n    cap = max(1, math.ceil(len(v) * cfg["bottom_fraction"]))\n    indices = indices[torch.argsort(confidence[indices], stable=True)[:cap]]\n    info = {"steps": len(v), "eligible_count": int(stable.sum()),\n            "selected_count": len(indices), "beta": 0.0, "coherence": None,\n            "fallback": None, "selected_indices": indices.cpu().tolist(),\n            "mean_abs_normal_logit_delta": 0.0}\n    if cfg["beta_max"] == 0:\n        info["fallback"] = "beta_zero_control"\n        return logits_a, info, None\n    if len(indices) < cfg["min_snippets"]:\n        info["fallback"] = "insufficient_confident_snippets"\n        return logits_a, info, None\n    weights = (1 - pc[indices]) * (1 - pa[indices])\n    pooled = (v[indices] * weights[:, None]).sum(0) / weights.sum()\n    norm = pooled.norm()\n    if norm <= 1e-8:\n        info["fallback"] = "degenerate_anchor"\n        return logits_a, info, None\n    anchor = F.normalize(pooled, dim=0)\n    coherence = float((v[indices] @ anchor).mean())\n    info["coherence"] = coherence\n    if coherence < cfg["min_coherence"]:\n        info["fallback"] = "incoherent_selected_features"\n        return logits_a, info, None\n    certainty = float((1 - confidence[indices]).mean())\n    support = min(1.0, len(indices) / max(cfg["min_snippets"], cap))\n    beta = cfg["beta_max"] * certainty * support * max(0.0, min(1.0, coherence))\n    native_normal = text[:, :, 0].float()\n    mixed = (1 - beta) * native_normal + beta * anchor[None, :]\n    if (mixed.norm(dim=-1) <= 1e-8).any():\n        info["fallback"] = "degenerate_mixture"\n        return logits_a, info, None\n    normal = F.normalize(mixed, dim=-1)\n    info["beta"] = beta\n    return logits_a, info, normal\n\n\n@torch.no_grad()\ndef anchor_forward(model, forward, images, mask, prompts, lengths, cfg, records):\n    """One native forward, followed by replacement of only its final Normal logit."""\n    if model.training:\n        raise ValueError("V16 requires model.eval() and frozen inference")\n    if not prompts or str(prompts[0]).strip().casefold() != "normal":\n        raise ValueError("Expected Normal at class index 0 in native prompt order")\n    result = forward(images, mask, prompts, lengths)\n    visual, text, temperature = model._v16_capture\n    del model._v16_capture\n    ori, c, a = result\n    # This is also a smoke test of capture placement, dtype and native logit scale.\n    reconstructed = visual @ text / temperature\n    if not torch.equal(a, reconstructed):\n        raise ValueError("Captured embeddings do not reproduce native A logits exactly")\n    _, info, normal = adapt_normal(visual, text, c, a, lengths, cfg)\n    info["temperature"] = temperature\n    if normal is None:\n        changed = a.detach()\n    else:\n        # Use the native matmul shape/dtype; all anomaly columns remain bitwise unchanged.\n        modified_text = text.detach().clone()\n        modified_text[:, :, 0] = normal.to(text.dtype)\n        candidate = visual @ modified_text / temperature\n        changed = a.detach().clone()\n        sizes = torch.as_tensor(lengths, device=a.device).reshape(-1)\n        valid = torch.arange(a.shape[1], device=a.device)[None, :] < sizes[:, None]\n        changed[..., 0][valid] = candidate[..., 0][valid]\n        info["mean_abs_normal_logit_delta"] = float(\n            (changed[..., 0][valid] - a[..., 0][valid]).abs().mean())\n    if not torch.equal(changed[..., 1:], a[..., 1:]) or not changed.isfinite().all():\n        raise ValueError("V16 changed anomaly logits or produced nonfinite values")\n    records.append(info)\n    return ori.detach(), c.detach(), changed\n'


In [ ]:
ANALYSIS_SOURCE = '"""Paired, video-bootstrap metrics without treating correlated frames as iid."""\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\ndef ranked_counts(scores, positives, negatives):\n    order = np.argsort(-scores, kind="stable")\n    end = np.r_[np.flatnonzero(np.diff(scores[order]) != 0), len(order) - 1]\n    return order, end, positives[order], negatives[order]\n\n\ndef weighted_metrics(ranked, video_weights):\n    order, end, pos, neg = ranked\n    weights = video_weights[order]\n    tp = np.cumsum(pos * weights)[end]\n    fp = np.cumsum(neg * weights)[end]\n    if not len(tp) or tp[-1] <= 0 or fp[-1] <= 0:\n        return None\n    recall = tp / tp[-1]\n    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=tp + fp > 0)\n    ap = np.sum(np.diff(np.r_[0, recall]) * precision)\n    x, y = np.r_[0, fp / fp[-1]], np.r_[0, recall]\n    auc = np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2)\n    return float(auc), float(ap)\n\n\ndef paired_bootstrap(scores0, scores1, gt, offsets, repeats, seed=42):\n    positives = gt.reshape(-1, 16).sum(1).astype(float)\n    negatives = 16 - positives\n    ranks = [ranked_counts(s, positives, negatives) for s in (scores0, scores1)]\n    video_index = np.repeat(np.arange(len(offsets) - 1), np.diff(offsets))\n    rng = np.random.default_rng(seed)\n    deltas = []\n    for _ in range(repeats):\n        counts = rng.multinomial(len(offsets) - 1, np.full(len(offsets) - 1, 1 / (len(offsets) - 1)))\n        values = [weighted_metrics(r, counts[video_index]) for r in ranks]\n        if all(v is not None for v in values):\n            deltas.append(np.subtract(values[1], values[0]) * 100)\n    if not deltas:\n        return {"requested": repeats, "valid": 0, "auc_ci95_pp": None, "ap_ci95_pp": None}\n    bounds = np.quantile(deltas, [0.025, 0.975], axis=0)\n    return {"requested": repeats, "valid": len(deltas),\n            "auc_ci95_pp": bounds[:, 0].tolist(), "ap_ci95_pp": bounds[:, 1].tolist()}\n\n\ndef false_alarm_diagnostics(scores, gt, offsets):\n    """Retrospective test-label operating points, NOT deployable calibrated thresholds."""\n    frames = np.repeat(scores, 16)\n    truth = gt.astype(bool)\n    positive_video = np.zeros(len(gt), dtype=bool)\n    for a, b in zip(offsets[:-1] * 16, offsets[1:] * 16, strict=True):\n        positive_video[a:b] = truth[a:b].any()\n    result = []\n    for requested in (0.5, 0.8):\n        positive_scores = np.sort(frames[truth])[::-1]\n        threshold = float(positive_scores[max(0, int(np.ceil(requested * len(positive_scores))) - 1)])\n        detected = frames >= threshold\n        row = {"requested_recall": requested, "actual_recall": float(detected[truth].mean()),\n               "threshold": threshold, "status": "retrospective_test_diagnostic_only"}\n        for name, mask in (("all_background", ~truth),\n                           ("background_in_positive_videos", ~truth & positive_video),\n                           ("all_negative_videos", ~truth & ~positive_video)):\n            row[name + "_frames"] = int(mask.sum())\n            row[name + "_fp"] = int((detected & mask).sum())\n            row[name + "_fpr"] = float(detected[mask].mean()) if mask.any() else None\n        result.append(row)\n    return result\n\n\ndef score_arrays(scores, gt):\n    x = np.repeat(scores, 16)\n    return {"roc_auc": float(roc_auc_score(gt, x)),\n            "ap": float(average_precision_score(gt, x))}\n'


In [ ]:
UCF_RUNNER = 'def main():\n    COMMIT = "c41067f07d252efcda18008bea367886070c33b0"\n    UPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"\n    EXPECTED_INPUTS = {\n        "checkpoint_sha256": "d064aba05190ce5f6652151e6259b24ca6dda83a1e6289bf343afb6f905a2c79",\n        "split_sha256": "d621f4ccfb3e26274d215ad53b83fd1f596bb7e9f85cde69e01399db3da97c7f",\n        "ground_truth_sha256": "c6448801e0f73d19adf82506cd58d60a90c4f0913e49dcc1ad1d677ee12d6301",\n        "annotation_sha256": "82d581cd4685480ffda23eb0fe1bca441b672bfe99c457adca2df4cf4780c503",\n        "feature_inventory_sha256": "8aa3a3c58f9bbbcc8082e024f3892e014f059374e8b52ca737ae709945210026"\n    }\n\n    import csv\n    import hashlib\n    import importlib.util\n    import json\n    import os\n    import re\n    import subprocess\n    import sys\n    import tempfile\n    import time\n    from collections import defaultdict\n    from shutil import copyfile, copyfileobj, copytree\n    from urllib.error import URLError\n    from zipfile import BadZipFile, ZipFile\n\n    if not INPUT.is_dir():\n        raise FileNotFoundError("Import notebook trên Kaggle và gắn input trước.")\n\n    def git(*args):\n        return subprocess.check_output(["git", *map(str, args)], text=True).strip()\n\n\n    if not SRC_REPO.exists():\n        SRC_REPO.parent.mkdir(parents=True, exist_ok=True)\n        with tempfile.TemporaryDirectory(dir=SRC_REPO.parent) as checkout_dir:\n            checkout = Path(checkout_dir) / "source"\n            subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)\n            subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)\n            checkout.rename(SRC_REPO)\n    if git("-C", SRC_REPO, "rev-parse", "HEAD") != COMMIT:\n        raise RuntimeError(f"Mã nguồn tại {SRC_REPO} sai commit. Hãy dùng thư mục nguồn mới.")\n    if git("-C", SRC_REPO, "status", "--porcelain", "--untracked-files=no"):\n        raise RuntimeError(f"Mã nguồn tại {SRC_REPO} có thay đổi. Hãy dùng checkout sạch.")\n\n    packages = {\n        "ftfy": "ftfy", "packaging": "packaging", "regex": "regex",\n        "tqdm": "tqdm", "sklearn": "scikit-learn",\n    }\n    missing_packages = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]\n    if missing_packages:\n        subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)\n\n    np = importlib.import_module("numpy")\n\n    # Modern setuptools no longer ships pkg_resources. Adapt the import in a copy.\n    stage_directory = tempfile.TemporaryDirectory(prefix="vadclip_ucf_v5_")\n    staging = Path(stage_directory.name)\n    runtime_src = staging / "runtime"\n    copytree(SRC_REPO / "src", runtime_src, dirs_exist_ok=True)\n    from anchor import anchor_forward, patch_model_file\n    model_receipt = patch_model_file(runtime_src / \'model.py\')\n    anchor_records = []\n\n    clip_py = runtime_src / "clip" / "clip.py"\n    clip_source = clip_py.read_text(encoding="utf-8")\n    old_import = "from pkg_resources import packaging"\n    if clip_source.count(old_import) != 1:\n        raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích. Kiểm commit.")\n    clip_py.write_text(clip_source.replace(old_import, "import packaging.version"), encoding="utf-8")\n\n    OUTPUT.mkdir(parents=True, exist_ok=True)\n    print("Commit VadCLIP", COMMIT)\n    print(f"Chấm UCF-Crime: {VARIANT}")\n\n    CLASSES = (\n        "Normal", "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",\n        "Fighting", "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism",\n    )\n    FRAMES_PER_FEATURE = 16\n    FEATURE_DIM = 512\n    VIDEO_COUNT = 290\n    CLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"\n\n\n    def sha256(path):\n        with path.open("rb") as stream:\n            return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\n    def read_split(list_root):\n        """Đọc theo đúng thứ tự repo. Tính khoảng [start, end) theo frame gốc."""\n        with (list_root / "ucf_CLIP_rgbtest.csv").open(newline="", encoding="utf-8") as stream:\n            entries = list(csv.DictReader(stream))\n        annotations = {}\n        for line in (list_root / "Temporal_Anomaly_Annotation.txt").read_text().splitlines():\n            if not line.strip():\n                continue\n            name, label, *bounds = line.split()\n            video_id = Path(name).stem\n            if len(bounds) != 4 or video_id in annotations:\n                raise ValueError(f"Annotation không hợp lệ: {video_id}")\n            intervals = []\n            for start, end in zip(bounds[::2], bounds[1::2]):\n                lo, hi = int(start), int(end)\n                if (lo, hi) == (-1, -1):\n                    continue\n                if not 0 <= lo < hi:\n                    raise ValueError(f"Khoảng frame không hợp lệ: {video_id} {(lo, hi)}")\n                intervals.append((lo, hi))\n            annotations[video_id] = (label, intervals)\n\n        rows = []\n        for entry in entries:\n            feature_name = Path(entry["path"]).name\n            match = re.fullmatch(r"(.+)__([0-9])\\.npy", feature_name)\n            if match is None:\n                raise ValueError(f"Tên feature sai: {feature_name}")\n            video_id, crop = match.groups()\n            label, intervals = annotations[video_id]\n            if label != entry["label"] or label not in CLASSES:\n                raise ValueError(f"Nhãn không khớp giữa CSV và annotation: {video_id}")\n            rows.append({\n                "id": video_id, "label": label, "feature_name": feature_name,\n                "crop": int(crop), "intervals": intervals,\n            })\n        if len(rows) != VIDEO_COUNT or len({row["id"] for row in rows}) != VIDEO_COUNT:\n            raise ValueError("Cần đúng 290 ID duy nhất từ split UCF-Crime chính thức.")\n        return rows\n\n\n\n    def stage_member(archive, member, destination):\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        with archive.open(member) as source, destination.open("wb") as target:\n            copyfileobj(source, target, length=1024 * 1024)\n        return destination\n\n\n    def find_inputs(root, keys):\n        """Tìm theo tên file chính thức, kể cả khi Kaggle giữ ZIP nguyên gói."""\n        found = defaultdict(list)\n        if root.is_file():\n            archive_path = root\n        elif root.is_dir():\n            for folder, _, filenames in os.walk(root):\n                for name in filenames:\n                    key = name.casefold()\n                    if key in keys:\n                        found[key].append(Path(folder) / name)\n            if found:\n                return found\n            archives = sorted(root.rglob("ucfclipfeatures_test290.zip"))\n            if len(archives) != 1:\n                raise RuntimeError(\n                    "Gắn Dataset feature UCF-Crime của V0 hoặc chỉ định DATA_ROOT."\n                )\n            archive_path = archives[0]\n        else:\n            raise FileNotFoundError(f"Không thấy DATA_ROOT: {root}")\n\n        with ZipFile(archive_path) as archive:\n            for index, member in enumerate(archive.infolist()):\n                name = Path(member.filename).name\n                key = name.casefold()\n                if not member.is_dir() and key in keys:\n                    destination = staging / "features" / str(index) / name\n                    found[key].append(stage_member(archive, member, destination))\n        return found\n\n\n    def checkpoint_candidates():\n        if CHECKPOINT is not None:\n            path = Path(CHECKPOINT)\n            if not path.is_file():\n                raise FileNotFoundError(f"UCF_CHECKPOINT không trỏ đến file đang gắn: {path}")\n            return [path]\n\n        candidates = []\n        for path in INPUT.rglob("*"):\n            if path.suffix.lower() not in {".pth", ".zip"}:\n                continue\n            name = path.stem.casefold()\n            is_ucf_checkpoint = name.startswith("model_ucf") or ("ucf" in name and "checkpoint" in name)\n            if is_ucf_checkpoint and path.is_file():\n                candidates.append(path)\n        return sorted(candidates, key=lambda path: (path.suffix.lower() == ".zip", str(path)))\n\n\n    def choose_checkpoint():\n        expected = EXPECTED_INPUTS["checkpoint_sha256"]\n        candidates = checkpoint_candidates()\n        if not candidates:\n            raise FileNotFoundError(\n                "Không thấy checkpoint UCF trong /kaggle/input. Vào Add Input và gắn Dataset có "\n                "model_ucf.pth hoặc vadclip_ucf_checkpoint_upload.zip. Nếu tên khác, "\n                "đặt UCF_CHECKPOINT thành đường dẫn đầy đủ của file."\n            )\n\n        rejected = []\n        for path in candidates:\n            try:\n                if path.suffix.lower() == ".zip":\n                    with ZipFile(path) as archive:\n                        members = [\n                            item for item in archive.infolist()\n                            if not item.is_dir() and Path(item.filename).name.casefold() == "model_ucf.pth"\n                        ]\n                        for member in members:\n                            with archive.open(member) as stream:\n                                digest = hashlib.file_digest(stream, "sha256").hexdigest()\n                            if digest == expected:\n                                print(f"Checkpoint UCF V0 khớp SHA-256: {path} ({member.filename})")\n                                return stage_member(archive, member, staging / "checkpoint" / "model_ucf.pth")\n                    rejected.append(f"{path.name} không chứa model_ucf.pth khớp SHA-256")\n                elif sha256(path) == expected:\n                    print(f"Checkpoint UCF V0 khớp SHA-256: {path}")\n                    return path\n                else:\n                    rejected.append(f"{path.name} sai SHA-256")\n            except (BadZipFile, OSError) as exc:\n                rejected.append(f"{path.name}: {exc}")\n        raise ValueError("Không tìm được checkpoint UCF đúng hash V0. " + "; ".join(rejected))\n\n\n    def input_fingerprint(rows, list_root, checkpoint_path):\n        inventory = [[row["id"], row["label"], row["steps"], row["feature_sha256"]] for row in rows]\n        inventory_bytes = json.dumps(inventory, separators=(",", ":")).encode()\n        return {\n            "checkpoint_sha256": sha256(checkpoint_path),\n            "split_sha256": sha256(list_root / "ucf_CLIP_rgbtest.csv"),\n            "ground_truth_sha256": sha256(list_root / "gt_ucf.npy"),\n            "annotation_sha256": sha256(list_root / "Temporal_Anomaly_Annotation.txt"),\n            "feature_inventory_sha256": hashlib.sha256(inventory_bytes).hexdigest(),\n        }\n\n\n    def prepare_clip_cache(source, cache_dir):\n        """Đặt file OpenAI đã xác minh vào đúng cache mà loader gốc đọc."""\n        if not source.is_file():\n            raise FileNotFoundError(f"Không thấy CLIP_WEIGHTS: {source}")\n        if sha256(source) != CLIP_SHA256:\n            raise ValueError("CLIP_WEIGHTS sai SHA-256 của OpenAI ViT-B-16.pt.")\n        target = cache_dir / "ViT-B-16.pt"\n        if target.is_file() and sha256(target) == CLIP_SHA256:\n            return target\n        cache_dir.mkdir(parents=True, exist_ok=True)\n        temporary = cache_dir / "ViT-B-16.pt.vadclip-part"\n        copyfile(source, temporary)\n        if sha256(temporary) != CLIP_SHA256:\n            raise ValueError("Bản sao CLIP lỗi. Kiểm dung lượng và file input.")\n        temporary.replace(target)\n        return target\n\n\n    def feature_array(path):\n        array = np.load(path, mmap_mode="r", allow_pickle=False)\n        if array.ndim != 2 or array.shape[1] != FEATURE_DIM or array.shape[0] < 1:\n            raise ValueError(f"Feature cần dạng [T, 512], nhận {array.shape}: {path}")\n        if array.dtype not in (np.float16, np.float32, np.float64):\n            raise ValueError(f"Feature phải là số thực, nhận {array.dtype}: {path}")\n        return array\n\n\n    def frame_labels(row, length):\n        labels = np.zeros(length, dtype=np.uint8)\n        for start, end in row["intervals"]:\n            labels[start:end] = 1\n        return labels\n\n\n    def validate_ground_truth(rows, gt):\n        """Kiểm cả nội dung và thứ tự, ngoài phép kiểm tổng số frame."""\n        offset = 0\n        for row in rows:\n            stop = offset + row["steps"] * FRAMES_PER_FEATURE\n            expected = frame_labels(row, stop - offset)\n            if not np.array_equal(gt[offset:stop], expected):\n                raise ValueError(\n                    f"Ground truth lệch tại {row[\'id\']} (frame offset {offset}, "\n                    f"{row[\'steps\']} bước). Kiểm lại bản video/feature và frame count."\n                )\n            offset = stop\n        if offset != len(gt):\n            raise ValueError(f"Số frame không khớp: input={offset}, ground truth={len(gt)}")\n\n    # Kiểm đầu vào và đối chiếu hash với V0 đã chạy.\n    list_root = SRC_REPO / "list"\n    rows = read_split(list_root)\n    data_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT\n    expected = {row["feature_name"].casefold(): row["feature_name"] for row in rows}\n    found = find_inputs(data_root, expected)\n    missing = sorted(expected[key] for key in expected if not found[key])\n    ambiguous = {\n        expected[key]: [str(path) for path in sorted(found[key])]\n        for key in expected if len(found[key]) > 1\n    }\n    if missing or ambiguous:\n        issue_path = OUTPUT / "input_issues.json"\n        issue_path.write_text(json.dumps({\n            "missing": missing,\n            "ambiguous": ambiguous,\n        }, indent=2), encoding="utf-8")\n        raise RuntimeError(\n            f"Thiếu {len(missing)} file, trùng {len(ambiguous)} ID. "\n            f"Xem {issue_path} và kiểm DATA_ROOT."\n        )\n    for row in rows:\n        path = found[row["feature_name"].casefold()][0]\n        row["path"] = path\n        row["steps"] = len(feature_array(path))\n        row["feature_sha256"] = sha256(path)\n\n    gt = np.load(list_root / "gt_ucf.npy", allow_pickle=False)\n    if gt.ndim != 1 or set(np.unique(gt)) != {0.0, 1.0}:\n        raise ValueError("Ground truth UCF cần vector nhị phân cấp frame.")\n    validate_ground_truth(rows, gt)\n    checkpoint_path = choose_checkpoint()\n    input_hashes = input_fingerprint(rows, list_root, checkpoint_path)\n    mismatches = [key for key, value in EXPECTED_INPUTS.items() if input_hashes[key] != value]\n    if mismatches:\n        raise ValueError("Đầu vào khác lượt V0 đã đối chiếu: " + ", ".join(mismatches))\n    print(f"Đã khớp {len(rows)} video, {len(gt):,} frame và toàn bộ hash đầu vào với V0.")\n\n    import torch\n\n    sys.path.insert(0, str(runtime_src))\n    from model import CLIPVAD\n    from ucf_option import parser as ucf_parser\n    from utils.tools import get_batch_mask\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Hãy bật GPU trong Kaggle. Mã VadCLIP gốc cần CUDA khi chạy forward.")\n    device = torch.device("cuda:0")\n    torch.cuda.set_device(device)\n    torch.manual_seed(234)\n\n    # Dùng đúng cấu hình UCF từ mã tác giả, không đọc cờ của kernel Jupyter.\n    args = ucf_parser.parse_args([])\n    evaluated_checkpoint = checkpoint_path\n    evaluated_sha256 = sha256(evaluated_checkpoint)\n    state = torch.load(evaluated_checkpoint, map_location="cpu", weights_only=True)\n    if isinstance(state, dict):\n        for field in ("state_dict", "model_state_dict"):\n            if field in state:\n                state = state[field]\n                break\n    if CLIP_WEIGHTS is not None:\n        prepare_clip_cache(Path(CLIP_WEIGHTS), Path.home() / ".cache" / "clip")\n    try:\n        model = CLIPVAD(\n            args.classes_num, args.embed_dim, args.visual_length, args.visual_width, args.visual_head,\n            args.visual_layers, args.attn_window, args.prompt_prefix, args.prompt_postfix, device,\n        )\n    except URLError as exc:\n        raise RuntimeError(\n            "Không tải được CLIP ViT-B/16. Bật Internet hoặc gắn ViT-B-16.pt chính thức "\n            "và đặt CLIP_WEIGHTS bằng đường dẫn file."\n        ) from exc\n    try:\n        model.load_state_dict(state, strict=True)\n    except (RuntimeError, TypeError) as exc:\n        raise RuntimeError(\n            f"Checkpoint không khớp model VadCLIP UCF-Crime: {evaluated_checkpoint}. "\n            "Kiểm lại file từ cột Model dòng UCF-Crime, khác file ViT-B-16.pt."\n        ) from exc\n    del state\n    model.to(device).eval()\n    print("Đã nạp đầy đủ checkpoint trên", torch.cuda.get_device_name(device))\n\n    def predict_video(feature, model, device, block_length):\n        """Trả điểm [T, 2], lần lượt sigmoid(C) và 1-softmax(A)[Normal]."""\n        steps = len(feature)\n        if steps < 1 or not np.isfinite(feature).all():\n            raise ValueError("Feature rỗng hoặc chứa NaN/Inf.")\n        blocks = (steps + block_length - 1) // block_length\n        padded = np.zeros((blocks * block_length, FEATURE_DIM), dtype=np.float32)\n        padded[:steps] = feature\n        lengths = torch.tensor([\n            min(block_length, steps - i * block_length) for i in range(blocks)\n        ])\n        visual = torch.from_numpy(padded.reshape(blocks, block_length, FEATURE_DIM)).to(device)\n        mask = get_batch_mask(lengths, block_length).to(device)\n        with torch.no_grad():\n            if VARIANT == "v0":\n                _, logits_c, logits_a = model(visual, mask, list(CLASSES), lengths)\n            else:\n                _, logits_c, logits_a = anchor_forward(\n                    model, model, visual, mask, list(CLASSES), lengths,\n                    ANCHOR_CONFIG, anchor_records,\n                )\n            c_score = logits_c.reshape(-1)[:steps].sigmoid()\n            a_score = 1 - logits_a.reshape(-1, len(CLASSES))[:steps].softmax(dim=-1)[:, 0]\n            scores = torch.stack((c_score, a_score), dim=1).cpu().numpy()\n        if scores.shape != (steps, 2) or not np.isfinite(scores).all():\n            raise ValueError(f"Điểm bất thường không hợp lệ: {scores.shape}")\n        return scores\n\n\n    def run_video(row):\n        feature = feature_array(row["path"])\n        if feature.shape != (row["steps"], FEATURE_DIM):\n            raise ValueError(f"Feature {row[\'id\']} sai shape: {feature.shape}")\n        result = predict_video(feature, model, device, args.visual_length)\n        if VARIANT != "v0":\n            anchor_records[-1]["video_id"] = row["id"]\n        return result\n\n\n    # Chọn theo thứ tự split và độ dài, trước khi xem điểm.\n    pilot = sorted({\n        next(i for i, row in enumerate(rows) if row["label"] != "Normal"),\n        next(i for i, row in enumerate(rows) if row["label"] == "Normal"),\n        max(range(len(rows)), key=lambda i: rows[i]["steps"]),\n    })\n    torch.cuda.reset_peak_memory_stats(device)\n    started = time.perf_counter()\n    scores_by_index = {}\n    for i in pilot:\n        before = time.perf_counter()\n        scores_by_index[i] = run_video(rows[i])\n        print(rows[i]["id"], scores_by_index[i].shape, f"{time.perf_counter() - before:.2f}s")\n    smoke_seconds = time.perf_counter() - started\n    print(f"Smoke test hoàn tất trong {smoke_seconds:.2f}s.")\n\n    if not SMOKE_ONLY:\n        for i, row in enumerate(rows):\n            if i not in scores_by_index:\n                scores_by_index[i] = run_video(row)\n            if (i + 1) % 25 == 0 or i + 1 == len(rows):\n                print(f"Đã chạy {i + 1}/{len(rows)} video", flush=True)\n\n    from importlib.metadata import version\n\n    from sklearn.metrics import average_precision_score, roc_auc_score\n\n    SCORE_COLUMNS = ("classification", "alignment")\n\n\n\n\n    def score_benchmark(rows, snippet_scores, gt):\n        steps_per_video = np.asarray([row["steps"] for row in rows])\n        if snippet_scores.shape != (sum(steps_per_video), len(SCORE_COLUMNS)):\n            raise ValueError("Số điểm snippet hoặc cột không khớp.")\n        frame_scores = np.repeat(snippet_scores, FRAMES_PER_FEATURE, axis=0)\n        if len(frame_scores) != len(gt):\n            raise ValueError("Số điểm cấp frame không khớp ground truth.")\n        frame_mask = np.repeat(\n            [row["label"] != "Normal" for row in rows],\n            steps_per_video * FRAMES_PER_FEATURE,\n        )\n        metrics = {}\n        for column, name in enumerate(SCORE_COLUMNS):\n            scores = frame_scores[:, column]\n            metrics[name] = {\n                "roc_auc": float(roc_auc_score(gt, scores)),\n                "ap": float(average_precision_score(gt, scores)),\n                "ano_auc": float(roc_auc_score(gt[frame_mask], scores[frame_mask])),\n            }\n        return metrics\n\n\n    selected = sorted(scores_by_index)\n    if not SMOKE_ONLY and selected != list(range(len(rows))):\n        raise ValueError("Thiếu dự đoán. Cần chạy toàn bộ split trước khi chấm.")\n    selected_rows = [rows[i] for i in selected]\n    ordered = [scores_by_index[i] for i in selected]\n    if any(score.shape != (row["steps"], 2)\n           for row, score in zip(selected_rows, ordered, strict=True)):\n        raise ValueError("Số điểm từng video không khớp.")\n    steps_per_video = [row["steps"] for row in selected_rows]\n    scores = np.concatenate(ordered, axis=0)\n\n    output_dir = OUTPUT / ("smoke" if SMOKE_ONLY else "full")\n    output_dir.mkdir(parents=True, exist_ok=True)\n    metadata = {\n        "run_kind": "smoke_inference" if SMOKE_ONLY else "full_checkpoint_inference",\n        "benchmark_scope": "ucf_crime_official_test",\n        "evaluation_status": "exploratory_on_previously_seen_official_test",\n        "commit": COMMIT,\n        "runtime_patch": "CLIP imports packaging.version directly",\n        **input_hashes,\n        "clip_backbone_sha256": CLIP_SHA256,\n        "frames_per_feature": FRAMES_PER_FEATURE,\n        "visual_block_length": args.visual_length,\n        "label_order": CLASSES,\n        "score_columns": SCORE_COLUMNS,\n        "evaluated_checkpoint_sha256": evaluated_sha256,\n        "python": sys.version.split()[0],\n        "libraries": {name: version(name) for name in (\n            "torch", "torchvision", "numpy", "scipy", "scikit-learn", "pillow",\n        )},\n        "cuda": torch.version.cuda,\n        "gpu": torch.cuda.get_device_name(device),\n        "smoke_seconds": smoke_seconds,\n        "inference_seconds": time.perf_counter() - started,\n        "peak_torch_vram_mib": torch.cuda.max_memory_allocated(device) / (1024 ** 2),\n        "video_count": len(selected_rows),\n        "videos": [{key: value for key, value in row.items() if key != "path"}\n                   for row in selected_rows],\n        "metrics": None, "anchor_config": ANCHOR_CONFIG, **model_receipt,\n    }\n    if not SMOKE_ONLY:\n        metrics = score_benchmark(rows, scores, gt)\n        metadata["metrics"] = metrics\n        print(json.dumps(metrics, indent=2))\n\n    np.savez_compressed(\n        output_dir / "predictions.npz",\n        video_id=np.asarray([row["id"] for row in selected_rows]),\n        video_label=np.asarray([row["label"] for row in selected_rows]),\n        snippet_offsets=np.cumsum([0] + steps_per_video),\n        score_columns=np.asarray(SCORE_COLUMNS),\n        scores=scores,\n        frame_gt=gt if not SMOKE_ONLY else np.empty(0, dtype=np.uint8),\n    )\n    (output_dir / "run.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    (output_dir / "anchor_diagnostics.json").write_text(\n        json.dumps(anchor_records, indent=2), encoding="utf-8")\n    stage_directory.cleanup()\n    print("Đầu ra", output_dir)\n\nmain()\n'


In [ ]:
XD_RUNNER = 'def main():\n    COMMIT = "c41067f07d252efcda18008bea367886070c33b0"\n    UPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"\n    VIDEO_COUNT = 800\n    FRAMES_PER_FEATURE = 16\n    FRAME_COUNT = 2_331_296\n    FEATURE_DIM = 512\n    PAPER_AP = 0.8451\n    PACK_MANIFEST_NAME = "xd_test_manifest.json"\n    PACK_ARCHIVE_NAME = "vadclip_xd_test800.zip"\n    PREVIOUS_PACK_ARCHIVE_NAME = "vadclip_xd_test800_kaggle_v2.zip"\n    PACK_MANIFEST_SHA256 = "8e5d329708857338fa94fb63a1d5733d9d8536250e6c09c3ba012f54dce997b2"\n    CLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"\n\n    import csv\n    import hashlib\n    import importlib.util\n    import json\n    import os\n    import shutil\n    import subprocess\n    import sys\n    import tempfile\n    from collections import defaultdict\n    from contextlib import ExitStack\n    from pathlib import PurePosixPath\n    from zipfile import BadZipFile, ZipFile\n\n    if RUN_MODE not in {"smoke", "full"}:\n        raise ValueError("Evaluator XD cần chế độ smoke hoặc full.")\n    if not INPUT.is_dir():\n        raise FileNotFoundError("Gắn dữ liệu qua Add Input trong Kaggle trước khi chạy.")\n\n    OUTPUT.mkdir(parents=True, exist_ok=True)\n    scratch = SRC_REPO.parent\n    scratch.mkdir(parents=True, exist_ok=True)\n    src_repo = SRC_REPO\n    if not src_repo.exists():\n        with tempfile.TemporaryDirectory(dir=scratch) as checkout_dir:\n            checkout = Path(checkout_dir) / "source"\n            subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)\n            subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)\n            checkout.rename(src_repo)\n\n    def git(*args):\n        return subprocess.check_output(["git", "-C", str(src_repo), *args], text=True).strip()\n\n    if git("rev-parse", "HEAD") != COMMIT or git("status", "--porcelain", "--untracked-files=all"):\n        raise RuntimeError(f"Cần checkout sạch tại commit {COMMIT}. Kiểm tra {src_repo}.")\n\n    np = importlib.import_module("numpy")\n\n    def sha256(path):\n        with path.open("rb") as stream:\n            return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\n    def zip_members(path):\n        try:\n            with ZipFile(path) as archive:\n                return [name for name in archive.namelist() if not name.endswith("/")]\n        except BadZipFile as exc:\n            raise ValueError(f"ZIP chưa tải đủ hoặc bị hỏng: {path.name}") from exc\n\n\n    def basename(value):\n        return PurePosixPath(str(value).replace(chr(92), "/")).name\n\n\n    def read_split(path):\n        with path.open(newline="", encoding="utf-8-sig") as stream:\n            reader = csv.DictReader(stream)\n            if reader.fieldnames is None or not {"path", "label"} <= set(reader.fieldnames):\n                raise ValueError("CSV tác giả cần cột path và label.")\n            fields, rows = reader.fieldnames, list(reader)\n        source_paths = [row["path"] for row in rows]\n        if (\n            len(rows) != VIDEO_COUNT\n            or len(set(source_paths)) != VIDEO_COUNT\n            or any(not path or not basename(path).endswith("__0.npy") for path in source_paths)\n            or any(not row["label"] for row in rows)\n        ):\n            raise ValueError(f"Split XD phải gồm {VIDEO_COUNT} dòng feature hợp lệ, duy nhất.")\n        return fields, rows\n\n\n    def matching_files(root, expected):\n        found = defaultdict(list)\n        if FEATURE_ZIP is None:\n            if not root.is_dir():\n                raise FileNotFoundError(f"DATA_ROOT không tồn tại: {root}")\n            for path in root.rglob("*.npy"):\n                if path.name in expected:\n                    found[path.name].append((path, None))\n            if found:\n                return found\n            archives = sorted(root.rglob("*.zip"))\n        else:\n            archives = [Path(FEATURE_ZIP)]\n\n        for archive_path in archives:\n            if not archive_path.is_file():\n                raise FileNotFoundError(f"Không thấy FEATURE_ZIP: {archive_path}")\n            for member in zip_members(archive_path):\n                name = basename(member)\n                if name in expected:\n                    found[name].append((archive_path, member))\n        return found\n\n\n    def match_one(original, candidates):\n        # Nếu hai file cùng tên, chỉ phân giải khi hậu tố thư mục chính thức khớp duy nhất.\n        if len(candidates) <= 1:\n            return candidates\n        official = PurePosixPath(original.replace(chr(92), "/")).parts\n        for depth in range(2, len(official) + 1):\n            matches = [\n                (path, member)\n                for path, member in candidates\n                if PurePosixPath((member or path.as_posix()).replace(chr(92), "/")).parts[-depth:]\n                == official[-depth:]\n            ]\n            if len(matches) == 1:\n                return matches\n            if not matches:\n                break\n        return candidates\n\n\n    def pick_checkpoint(root):\n        if CHECKPOINT is not None:\n            path = Path(CHECKPOINT)\n            if not path.is_file() or path.suffix.lower() not in {".pth", ".zip"}:\n                raise FileNotFoundError(f"CHECKPOINT cần đường dẫn .pth hoặc .zip hợp lệ: {path}")\n            return path\n        weights = [path for path in root.rglob("*.pth") if path.name in {"model_xd.pth", "model.pth"}]\n        if len(weights) == 1:\n            return weights[0]\n        if not weights:\n            archives = [\n                path for path in root.rglob("*.zip")\n                if any(\n                    basename(member) in {"model_xd.pth", "model.pth"}\n                    for member in zip_members(path)\n                )\n            ]\n            if len(archives) == 1:\n                return archives[0]\n        raise RuntimeError(\n            "Gắn checkpoint VadCLIP XD-Violence. Nếu tên khác hoặc có nhiều bản, "\n            "đặt CHECKPOINT bằng đường dẫn file .pth hoặc ZIP chỉ chứa một .pth."\n        )\n\n\n    def stage_member(archive, member, target):\n        # Chỉ ghi tới tên đích do notebook tạo. Không giải nén theo đường dẫn trong ZIP.\n        target.parent.mkdir(parents=True, exist_ok=True)\n        with archive.open(member) as src, target.open("wb") as dest:\n            shutil.copyfileobj(src, dest, length=1024 * 1024)\n        return target\n\n\n    def checkpoint_from_input(root, staging):\n        path = pick_checkpoint(root)\n        if path.suffix.lower() == ".zip":\n            members = [name for name in zip_members(path) if name.lower().endswith(".pth")]\n            if len(members) != 1:\n                raise ValueError(f"ZIP checkpoint phải chứa đúng một .pth: {path.name}")\n            if basename(members[0]) == "model_ucf.pth":\n                raise ValueError("ZIP đang chứa checkpoint UCF-Crime.")\n            with ZipFile(path) as archive:\n                path = stage_member(archive, members[0], staging / "checkpoint.pth")\n        if path.name == "model_ucf.pth" or path.stat().st_size == 0:\n            raise ValueError(f"File checkpoint không hợp lệ cho XD: {path.name}")\n        with path.open("rb") as stream:\n            if stream.read(4) == b"PK\\x03\\x04":\n                zip_members(path)  # xác nhận có thư mục ZIP ở cuối file PyTorch\n        return path\n\n\n    def feature_info(path):\n        array = np.load(path, mmap_mode="r", allow_pickle=False)\n        if (\n            array.ndim != 2 or array.shape[0] == 0 or array.shape[1] != FEATURE_DIM\n            or array.dtype not in (np.float16, np.float32, np.float64)\n            or not np.isfinite(array).all()\n        ):\n            raise ValueError(f"Feature cần mảng số thực hữu hạn [T, {FEATURE_DIM}]: {path.name}")\n        return len(array), sha256(path)\n\n\n    def read_pack_manifest(root):\n        candidates = [path.read_bytes() for path in root.rglob(PACK_MANIFEST_NAME)]\n        archives = set(root.rglob(PACK_ARCHIVE_NAME)) | set(root.rglob(PREVIOUS_PACK_ARCHIVE_NAME))\n        if FEATURE_ZIP is not None:\n            archives.add(Path(FEATURE_ZIP))\n        for archive_path in sorted(archives):\n            with ZipFile(archive_path) as archive:\n                members = [\n                    name for name in archive.namelist() if basename(name) == PACK_MANIFEST_NAME\n                ]\n                candidates.extend(archive.read(name) for name in members)\n        if not candidates:\n            raise FileNotFoundError(\n                "Chưa thấy manifest. Gắn Dataset VadCLIP XD-Violence Test 800 "\n                f"từ {PACK_ARCHIVE_NAME} và giữ nguyên tên file ZIP."\n            )\n        if any(hashlib.sha256(raw).hexdigest() != PACK_MANIFEST_SHA256 for raw in candidates):\n            raise ValueError("Manifest trên Kaggle không khớp bản đã khóa trong notebook.")\n\n        manifest = json.loads(candidates[0])\n        if manifest["upstream_commit"] != COMMIT:\n            raise ValueError("Commit trong manifest không đúng.")\n        return manifest\n\n\n    def check_pack_manifest(manifest, inventory, checkpoint_hash):\n        if manifest["checkpoint_sha256"] != checkpoint_hash:\n            raise ValueError("Sai checkpoint so với manifest đã khóa.")\n        actual_features = {\n            basename(row["official_path"]): {"steps": row["steps"], "sha256": row["feature_sha256"]}\n            for row in inventory\n        }\n        if actual_features != manifest["features"]:\n            names = actual_features.keys() | manifest["features"].keys()\n            first_difference = next(\n                name for name in sorted(names)\n                if actual_features.get(name) != manifest["features"].get(name)\n            )\n            raise ValueError(f"Feature trên Kaggle lệch bản đã khóa: {first_difference}")\n        return manifest["source_features_sha256"]\n\n\n    def prepare_inputs(split, gt_path, data_root, staging, upload_names=None):\n        fields, rows = read_split(split)\n        original_names = {basename(row["path"]) for row in rows}\n        if upload_names is not None and (\n            len(original_names) != VIDEO_COUNT\n            or upload_names.keys() != original_names\n            or len(set(upload_names.values())) != VIDEO_COUNT\n        ):\n            raise ValueError("Ánh xạ tên file không khớp danh sách video trong CSV gốc.")\n        names = upload_names if upload_names is not None else {name: name for name in original_names}\n        found = matching_files(data_root, set(names.values()))\n        resolved = []\n        issues = {"missing": [], "ambiguous": []}\n        for row in rows:\n            original = row["path"]\n            candidates = match_one(original, found[names[basename(original)]])\n            if len(candidates) == 0:\n                issues["missing"].append(original)\n            elif len(candidates) > 1:\n                issues["ambiguous"].append({\n                    "expected": original,\n                    "candidates": [str(path) if member is None else f"{path.name}::{member}"\n                                   for path, member in candidates],\n                })\n            else:\n                resolved.append(candidates[0])\n        if any(issues.values()) or len(set(resolved)) != VIDEO_COUNT:\n            report = OUTPUT / "input_issues.json"\n            report.write_text(json.dumps(issues, ensure_ascii=False, indent=2), encoding="utf-8")\n            raise RuntimeError(\n                f"Thiếu {len(issues[\'missing\'])} file, trùng {len(issues[\'ambiguous\'])} dòng "\n                f"hoặc dùng lại cùng feature. Xem {report}. Kiểm DATA_ROOT/FEATURE_ZIP."\n            )\n\n        gt = np.load(gt_path, allow_pickle=False)\n        if gt.shape != (FRAME_COUNT,) or not np.array_equal(np.unique(gt), [0, 1]):\n            raise ValueError("Ground truth XD sai shape hoặc không phải vector nhị phân.")\n\n        inventory = []\n        with ExitStack() as stack:\n            archives = {}\n            for index, (row, (path, member)) in enumerate(zip(rows, resolved, strict=True)):\n                if member is not None:\n                    if path not in archives:\n                        archives[path] = stack.enter_context(ZipFile(path))\n                    target = staging / f"{index:04d}" / basename(member)\n                    path = stage_member(archives[path], member, target)\n                count, digest = feature_info(path)\n                inventory.append({\n                    "official_path": row["path"], "label": row["label"],\n                    "steps": count, "feature_sha256": digest,\n                })\n                row["path"] = str(path)\n        steps = sum(item["steps"] for item in inventory)\n        if steps * FRAMES_PER_FEATURE != len(gt):\n            raise ValueError(\n                f"Feature có {steps:,} bước, cần {len(gt) // FRAMES_PER_FEATURE:,} "\n                f"theo ground truth tác giả. Kiểm phiên bản gói feature và crop."\n            )\n\n        offset = 0\n        for item in inventory:\n            stop = offset + item["steps"] * FRAMES_PER_FEATURE\n            positive_frames = int(np.count_nonzero(gt[offset:stop]))\n            if "_label_A" in basename(item["official_path"]) and positive_frames:\n                raise ValueError(f"Lệch ground truth trong video Normal: {item[\'official_path\']}")\n            item["positive_frames"] = positive_frames\n            offset = stop\n\n        resolved_csv = staging / "xd_CLIP_rgbtest_kaggle.csv"\n        with resolved_csv.open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n        return resolved_csv, inventory, gt\n\n    # Preflight CPU. Mỗi lần chạy tạo thư mục mới, không ghi đè input đã gắn.\n    stage_directory = tempfile.TemporaryDirectory(prefix="run_", dir=scratch)\n    staging = Path(stage_directory.name)\n    split = src_repo / "list" / "xd_CLIP_rgbtest.csv"\n    gt_path = src_repo / "list" / "gt.npy"\n    data_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT\n    try:\n        pack_manifest = read_pack_manifest(INPUT) if REQUIRE_PACK_MANIFEST else None\n        upload_names = pack_manifest["upload_names"] if pack_manifest is not None else None\n        resolved_csv, inventory, gt = prepare_inputs(split, gt_path, data_root, staging, upload_names)\n        checkpoint_path = checkpoint_from_input(INPUT, staging)\n        checkpoint_hash = sha256(checkpoint_path)\n        source_features_hash = None\n        if pack_manifest is not None:\n            source_features_hash = check_pack_manifest(pack_manifest, inventory, checkpoint_hash)\n    except Exception:\n        stage_directory.cleanup()\n        raise\n\n    receipt = {\n        "upstream_commit": COMMIT,\n        "split_sha256": sha256(split),\n        "ground_truth_sha256": sha256(gt_path),\n        "checkpoint_sha256": checkpoint_hash,\n        "pack_manifest_sha256": PACK_MANIFEST_SHA256 if REQUIRE_PACK_MANIFEST else None,\n        "source_features_archive_sha256": source_features_hash,\n        "video_count": len(inventory),\n        "snippet_count": sum(row["steps"] for row in inventory),\n        "frame_count": len(gt),\n        "frames_per_feature": FRAMES_PER_FEATURE,\n        "videos": inventory,\n    }\n    if REFERENCE_MANIFEST is not None:\n        try:\n            reference = json.loads(Path(REFERENCE_MANIFEST).read_text(encoding="utf-8"))\n            for key in ("upstream_commit", "split_sha256", "ground_truth_sha256",\n                        "checkpoint_sha256", "videos"):\n                if receipt[key] != reference.get(key):\n                    raise ValueError(f"Bản Kaggle lệch bản tham chiếu ở trường {key}.")\n        except Exception:\n            stage_directory.cleanup()\n            raise\n    receipt["matched_reference_manifest"] = REFERENCE_MANIFEST is not None\n\n    (OUTPUT / "preflight.json").write_text(\n        json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(\n        f"Preflight đạt: {len(inventory)} video, {receipt[\'snippet_count\']:,} bước, "\n        f"{len(gt):,} frame."\n    )\n    print(f"Checkpoint {checkpoint_path.name}, SHA-256 {checkpoint_hash}")\n    if REQUIRE_PACK_MANIFEST:\n        print("SHA-256: khớp cả 800 feature và checkpoint công bố với hai gói đã chuẩn bị.")\n\n    receipt["evaluated_checkpoint_sha256"] = checkpoint_hash\n\n    import contextlib\n    import runpy\n    import time\n    import traceback\n    from unittest.mock import patch\n\n\n\n\n    def run_upstream():\n        full_run = RUN_MODE == "full"\n        packages = {\n            "ftfy": "ftfy", "packaging": "packaging", "regex": "regex", "tqdm": "tqdm",\n            "sklearn": "scikit-learn", "scipy": "scipy", "pandas": "pandas",\n        }\n        missing = [\n            package for module, package in packages.items()\n            if importlib.util.find_spec(module) is None\n        ]\n        if missing:\n            subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)\n\n        import torch\n        from sklearn import metrics\n\n        if not torch.cuda.is_available():\n            raise RuntimeError("Bật Accelerator GPU T4 trong Kaggle để chạy inference.")\n        torch.cuda.set_device(0)\n        torch.manual_seed(234)\n        runtime = staging / "runtime"\n        shutil.copytree(src_repo / "src", runtime)\n        from anchor import anchor_forward, patch_model_file\n        model_receipt = patch_model_file(runtime / "model.py")\n        anchor_records = []\n\n        clip_source = runtime / "clip" / "clip.py"\n        text = clip_source.read_text(encoding="utf-8")\n        old_import = "from pkg_resources import packaging"\n        if text.count(old_import) != 1:\n            raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích tại commit đã khóa.")\n        clip_source.write_text(text.replace(old_import, "import packaging.version"), encoding="utf-8")\n\n        if CLIP_WEIGHTS is not None:\n            source = Path(CLIP_WEIGHTS)\n            if not source.is_file() or sha256(source) != CLIP_SHA256:\n                raise ValueError("CLIP_WEIGHTS cần file OpenAI ViT-B-16.pt đúng SHA-256.")\n            cache = Path.home() / ".cache" / "clip"\n            cache.mkdir(parents=True, exist_ok=True)\n            target = cache / "ViT-B-16.pt"\n            if source.resolve() != target.resolve():\n                with tempfile.TemporaryDirectory(dir=cache) as copy_dir:\n                    temporary = Path(copy_dir) / target.name\n                    shutil.copyfile(source, temporary)\n                    if sha256(temporary) != CLIP_SHA256:\n                        raise ValueError("Bản sao trọng số CLIP không khớp SHA-256.")\n                    temporary.replace(target)\n\n        output_root = OUTPUT / RUN_MODE\n        output_root.mkdir(exist_ok=True)\n        output_dir = Path(tempfile.mkdtemp(prefix="run_", dir=output_root))\n        print(f"Đang chạy evaluator gốc. Log: {output_dir / \'upstream.log\'}", flush=True)\n        native_ap = metrics.average_precision_score\n        ap_calls = []\n        upstream_completed = False\n        post_ap_error = None\n        checkpoint_loaded = False\n        forwards = 0\n        torch_load = torch.load\n\n        class SmokeComplete(Exception):\n            pass\n\n        def load_checkpoint(file, *args, **kwargs):\n            nonlocal checkpoint_loaded\n            if not isinstance(file, (str, os.PathLike)) or Path(file).resolve() != checkpoint_path.resolve():\n                return torch_load(file, *args, **kwargs)\n            checkpoint_loaded = True\n            state = torch_load(file, map_location="cpu", weights_only=True)\n            for field in ("state_dict", "model_state_dict"):\n                if isinstance(state, dict) and field in state:\n                    return state[field]\n            return state\n\n        def record_ap(y_true, y_score, *args, **kwargs):\n            value = native_ap(y_true, y_score, *args, **kwargs)\n            if np.asarray(y_true).shape == gt.shape and np.array_equal(y_true, gt):\n                scores = np.asarray(y_score).reshape(-1)\n                if scores.shape != gt.shape or not np.isfinite(scores).all():\n                    raise ValueError("Điểm frame từ evaluator XD sai shape hoặc chứa NaN/Inf.")\n                grouped = scores.reshape(-1, FRAMES_PER_FEATURE)\n                if not np.all(grouped == grouped[:, :1]):\n                    raise ValueError("Điểm từ evaluator không theo quy tắc lặp 16 frame.")\n                ap_calls.append((float(value), grouped[:, 0].copy()))\n            return value\n\n        original_cwd = Path.cwd()\n        original_path = sys.path.copy()\n        original_argv = sys.argv.copy()\n        original_bytecode = sys.dont_write_bytecode\n        try:\n            os.chdir(runtime)\n            sys.path.insert(0, str(runtime))\n            sys.dont_write_bytecode = True\n            from model import CLIPVAD\n            from xd_option import parser\n\n            # Giữ các giá trị mặc định XD của tác giả, chỉ thay đường dẫn input.\n            destinations = {action.dest for action in parser._actions}\n            overrides = {\n                "test_list": str(resolved_csv),\n                "gt_path": str(gt_path),\n                "model_path": str(checkpoint_path),\n            }\n            defaults = parser.parse_args([])\n            for key in ("gt_segment_path", "gt_label_path"):\n                default = Path(getattr(defaults, key))\n                path = (src_repo / "src" / default).resolve()\n                if not path.is_file():\n                    path = (src_repo / default).resolve()\n                relative = path.relative_to(src_repo)\n                git("ls-files", "--error-unmatch", relative.as_posix())\n                overrides[key] = str(path)\n            if not overrides.keys() <= destinations:\n                raise RuntimeError(f"CLI upstream thiếu tham số: {overrides.keys() - destinations}")\n            flags = []\n            for action in parser._actions:\n                if action.dest in overrides:\n                    flags.extend((action.option_strings[0], overrides[action.dest]))\n\n            original_forward = CLIPVAD.forward\n\n            smoke_forwards = max(3, next(\n                i for i, row in enumerate(inventory, 1) if row["steps"] > defaults.visual_length\n            ))\n\n            def count_forward(self, images, padding_mask, text, lengths):\n                nonlocal forwards\n\n                def forward(x, mask, prompts, sizes):\n                    return original_forward(self, x, mask, prompts, sizes)\n\n                if VARIANT == "v0":\n                    result = forward(images, padding_mask, text, lengths)\n                else:\n                    result = anchor_forward(\n                        self, forward, images, padding_mask, text, lengths,\n                        ANCHOR_CONFIG, anchor_records,\n                    )\n                    anchor_records[-1]["video_id"] = inventory[forwards]["official_path"]\n                forwards += 1\n                if RUN_MODE == "smoke" and forwards == smoke_forwards:\n                    if any(not torch.isfinite(tensor).all() for tensor in result):\n                        raise ValueError("Smoke test sinh NaN/Inf.")\n                    raise SmokeComplete\n                return result\n\n            sys.argv = [str(runtime / "xd_test.py"), *flags]\n            torch.cuda.reset_peak_memory_stats()\n            started = time.perf_counter()\n            with (\n                (output_dir / "upstream.log").open("w", encoding="utf-8") as log,\n                contextlib.redirect_stdout(log),\n                contextlib.redirect_stderr(log),\n                patch.object(torch, "load", load_checkpoint),\n                patch.object(metrics, "average_precision_score", record_ap),\n                patch.object(CLIPVAD, "forward", count_forward),\n            ):\n                try:\n                    runpy.run_path(str(runtime / "xd_test.py"), run_name="__main__")\n                    upstream_completed = True\n                except SmokeComplete:\n                    if RUN_MODE != "smoke":\n                        raise\n                except Exception as exc:\n                    if not full_run or len(ap_calls) != 2:\n                        raise\n                    traceback.print_exc(file=log)\n                    post_ap_error = f"{type(exc).__name__}: {exc}"\n            elapsed = time.perf_counter() - started\n        finally:\n            os.chdir(original_cwd)\n            sys.argv = original_argv\n            sys.path[:] = original_path\n            sys.dont_write_bytecode = original_bytecode\n\n        if not checkpoint_loaded or forwards == 0:\n            raise RuntimeError("Chưa xác nhận upstream đã nạp checkpoint XD và chạy forward.")\n\n        result = {\n            **receipt,\n            "run_kind": "smoke_inference" if RUN_MODE == "smoke" else "full_checkpoint_inference",\n            "runtime_patch": "CLIP imports packaging.version directly",\n            "clip_backbone_sha256": CLIP_SHA256,\n            "forward_calls": forwards,\n            "visual_block_length": defaults.visual_length,\n            "upstream_completed": upstream_completed,\n            "post_ap_error": post_ap_error,\n            "gpu": torch.cuda.get_device_name(0),\n            "torch": torch.__version__,\n            "elapsed_seconds": elapsed,\n            "peak_torch_vram_mib": torch.cuda.max_memory_allocated() / 2**20,\n            "metrics": None, "anchor_config": ANCHOR_CONFIG, **model_receipt,\n        }\n        if full_run:\n            if len(ap_calls) != 2:\n                raise RuntimeError(\n                    f"Cần hai AP cấp frame từ xd_test.py, nhận {len(ap_calls)}. "\n                    f"Xem {output_dir / \'upstream.log\'}."\n                )\n            scores = np.column_stack([column for _, column in ap_calls])\n            result["paper_table_1_frame_ap"] = PAPER_AP\n            result["metrics"] = {\n                f"branch_{index + 1}": {\n                    "frame_ap": ap,\n                    "roc_auc": float(metrics.roc_auc_score(gt, np.repeat(column, FRAMES_PER_FEATURE))),\n                    "gap_vs_paper_percentage_points": 100 * (ap - PAPER_AP),\n                }\n                for index, (ap, column) in enumerate(ap_calls)\n            }\n            score_columns = ["branch_1", "branch_2"]\n            steps_per_video = [item["steps"] for item in inventory]\n            result["score_columns"] = score_columns\n            np.savez_compressed(\n                output_dir / "predictions.npz",\n                official_path=np.asarray([item["official_path"] for item in inventory]),\n                video_label=np.asarray([item["label"] for item in inventory]),\n                snippet_offsets=np.cumsum([0] + steps_per_video),\n                score_columns=np.asarray(score_columns),\n                scores=scores,\n                frame_gt=gt,\n            )\n        elif forwards != smoke_forwards:\n            raise RuntimeError("Smoke test chưa đủ lượt forward.")\n\n        (output_dir / "anchor_diagnostics.json").write_text(\n            json.dumps(anchor_records, indent=2), encoding="utf-8")\n        (output_dir / "run.json").write_text(\n            json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"\n        )\n        if full_run:\n            if post_ap_error is not None:\n                print(f"Script gốc lỗi sau khi tính đủ hai AP. Xem upstream.log: {post_ap_error}")\n            for name, values in result["metrics"].items():\n                print(\n                    f"{name}: AP {values[\'frame_ap\'] * 100:.4f}% "\n                    f"(chênh {values[\'gap_vs_paper_percentage_points\']:+.4f} điểm phần trăm)"\n                )\n        else:\n            print("Smoke đạt, gồm video dài hơn block.")\n        print(f"Đầu ra: {output_dir}")\n\n\n    try:\n        if RUN_MODE == "preflight":\n            print("Đã kiểm input. Chọn GPU và RUN_MODE = \'smoke\' để thử checkpoint.")\n        else:\n            run_upstream()\n    finally:\n        stage_directory.cleanup()\n\nmain()\n'


In [ ]:
# ruff: noqa: F821
"""Notebook driver; configuration and embedded sources are defined in earlier cells."""

import csv
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import tempfile
import time
from zipfile import ZIP_DEFLATED, ZipFile


def run_dataset(name, runner, config, scripts):
    header = ["from pathlib import Path"]
    for key, value in config.items():
        code = f"Path({str(value)!r})" if isinstance(value, Path) else repr(value)
        header.append(f"{key} = {code}")
    script = scripts / f"{name}.py"
    script.write_text("\n".join(header) + "\n" + runner, encoding="utf-8")
    print(f"\n--- {name} ---", flush=True)
    with (scripts.parent / f"{name}_console.log").open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            [sys.executable, "-u", str(script)], stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
            env={**os.environ, "PYTHONIOENCODING": "utf-8"},
        ) as process:
            for line in process.stdout:
                log.write(line)
                print(line, end="", flush=True)
            if process.wait() != 0:
                raise RuntimeError(f"{name} failed; see {name}_console.log")


V0_REFERENCE = {
    "ucf": ("classification", "roc_auc", 0.8801992605166816),
    "xd": ("branch_2", "frame_ap", 0.8450448809004643),
}


def compare(prefix, paths, analysis):
    import numpy as np

    reports = {key: json.loads(path.read_text(encoding="utf-8")) for key, path in paths.items()}
    reference = reports["v0"]
    expected = 290 if prefix == "ucf" else 800
    for report in reports.values():
        if report["run_kind"] != "full_checkpoint_inference" or report["video_count"] != expected:
            raise ValueError("Full inference report missing official videos")
        for key in ("split_sha256", "ground_truth_sha256", "evaluated_checkpoint_sha256",
                    "visual_block_length", "frames_per_feature", "original_model_sha256"):
            if report[key] != reference[key]:
                raise ValueError(f"Baseline/variant provenance mismatch: {key}")
    branch, metric, target = V0_REFERENCE[prefix]
    observed = reference["metrics"][branch][metric]
    if abs(observed - target) > 0.00005:
        raise ValueError(f"V0 parity failed: {observed} vs {target}; do not interpret V16")
    arrays = {}
    for variant, path in paths.items():
        with np.load(path.with_name("predictions.npz"), allow_pickle=False) as z:
            arrays[variant] = {k: z[k].copy() for k in z.files}
    old, identity, new = (arrays[k] for k in ("v0", "identity", "v16"))
    for changed in (identity, new):
        for key in ("snippet_offsets", "frame_gt", "score_columns", "video_label",
                    "video_id" if prefix == "ucf" else "official_path"):
            if not np.array_equal(old[key], changed[key]):
                raise ValueError(f"Prediction mapping mismatch: {key}")
    if not np.array_equal(old["scores"], identity["scores"]):
        raise ValueError("beta=0 control does not reproduce BOTH V0 branches exactly")
    if not np.array_equal(old["scores"][:, 0], new["scores"][:, 0]):
        raise ValueError("C-Branch changed, invalid A-only experiment")
    gt, offsets = old["frame_gt"].reshape(-1), old["snippet_offsets"]
    if offsets[-1] * 16 != len(gt):
        raise ValueError("Snippet/frame mapping mismatch")
    a0, a1 = old["scores"][:, 1], new["scores"][:, 1]
    ci = analysis.paired_bootstrap(a0, a1, gt, offsets, BOOTSTRAP_REPLICATES, SEED)
    rows = []
    # Report BOTH branches. UCF's primary C score is intentionally unchanged.
    for column, name in enumerate(("C", "A")):
        before = analysis.score_arrays(old["scores"][:, column], gt)
        after = analysis.score_arrays(new["scores"][:, column], gt)
        for measure in ("roc_auc", "ap"):
            bounds = ci["auc_ci95_pp" if measure == "roc_auc" else "ap_ci95_pp"]
            if name == "C":
                bounds = [0.0, 0.0]
            rows.append({"dataset": prefix, "branch": name, "metric": measure,
                         "v0_pct": 100 * before[measure], "v16_pct": 100 * after[measure],
                         "delta_pp": 100 * (after[measure] - before[measure]),
                         "ci95_low_pp": None if bounds is None else bounds[0],
                         "ci95_high_pp": None if bounds is None else bounds[1]})
    if prefix == "ucf":
        for name, native_name in (("C", "classification"), ("A", "alignment")):
            before = reference["metrics"][native_name]["ano_auc"]
            after = reports["v16"]["metrics"][native_name]["ano_auc"]
            rows.append({"dataset": prefix, "branch": name, "metric": "ano_auc",
                         "v0_pct": 100 * before, "v16_pct": 100 * after,
                         "delta_pp": 100 * (after - before),
                         "ci95_low_pp": None, "ci95_high_pp": None})
    false_alarms = []
    for variant, scores in (("v0_A", a0), ("v16_A", a1)):
        false_alarms.extend({"dataset": prefix, "variant": variant, **r}
                           for r in analysis.false_alarm_diagnostics(scores, gt, offsets))
    records = json.loads(paths["v16"].with_name("anchor_diagnostics.json").read_text())
    ids = [r["video_id"] for r in records]
    if len(ids) != expected or len(set(ids)) != expected:
        raise ValueError("Anchor diagnostics must contain exactly one record per video")
    return rows, false_alarms, {
        "identity_exact": True, "C_exact": True, "bootstrap": ci,
        "active_anchor_videos": sum(r["beta"] > 0 for r in records),
        "fallback_videos": sum(r["beta"] == 0 for r in records),
        "mean_beta": float(np.mean([r["beta"] for r in records])),
        "changed_A_snippets": int(np.count_nonzero(a0 != a1)),
    }


def write_csv(path, rows):
    if rows:
        with path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


def run_both():
    if RUN_MODE not in {"smoke", "full"}:
        raise ValueError("RUN_MODE must be smoke or full")
    if not DATASETS or len(set(DATASETS)) != len(DATASETS) or set(DATASETS) - {"ucf", "xd"}:
        raise ValueError("DATASETS must contain ucf and/or xd without duplicates")
    if type(BOOTSTRAP_REPLICATES) is not int or BOOTSTRAP_REPLICATES < 0:
        raise ValueError("Invalid bootstrap count")
    if not INPUT.is_dir():
        raise FileNotFoundError("Attach four original V0 test/checkpoint datasets")
    root = WORK / "vadclip_v16"
    root.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="run_", dir=root))
    scripts = output / "runtime_scripts"
    scripts.mkdir()
    for filename, source in (("anchor.py", ANCHOR_SOURCE), ("analysis.py", ANALYSIS_SOURCE)):
        (scripts / filename).write_text(source, encoding="utf-8")
    spec = importlib.util.spec_from_file_location("v16_analysis", scripts / "analysis.py")
    analysis = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(analysis)
    spec = importlib.util.spec_from_file_location("v16_anchor", scripts / "anchor.py")
    anchor = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(anchor)
    anchor.validate_config(ANCHOR_CONFIG)
    experiment = {
        "version": "v16_instance_adaptive_normal_anchor", "mode": RUN_MODE,
        "anchor_config": ANCHOR_CONFIG, "seed": SEED,
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "status": "fixed_exploratory_inference_on_previously_seen_official_test",
        "training": False, "online_causal": False,
        "selection": "no test labels used in anchor or parameter selection",
        "runtime_sha256": {n: hashlib.sha256(s.encode()).hexdigest() for n, s in (
            ("anchor", ANCHOR_SOURCE), ("analysis", ANALYSIS_SOURCE),
            ("ucf", UCF_RUNNER), ("xd", XD_RUNNER))},
    }
    (output / "experiment.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")
    print("Output:", output, flush=True)
    shared = {"INPUT": INPUT, "WORK": WORK, "CLIP_WEIGHTS": CLIP_WEIGHTS,
              "SRC_REPO": Path(tempfile.gettempdir()) / "vadclip_xd_c41067f" / "upstream"}
    jobs = {
        "ucf": (UCF_RUNNER, {"DATA_ROOT": UCF_DATA_ROOT, "CHECKPOINT": UCF_CHECKPOINT}),
        "xd": (XD_RUNNER, {"DATA_ROOT": XD_DATA_ROOT, "CHECKPOINT": XD_CHECKPOINT,
                            "FEATURE_ZIP": XD_FEATURE_ZIP, "REFERENCE_MANIFEST": None,
                            "REQUIRE_PACK_MANIFEST": True, "RUN_MODE": RUN_MODE}),
    }
    runs, metrics, alarms = {}, [], []
    started = time.perf_counter()
    for prefix in DATASETS:
        runner, config = jobs[prefix]
        entry = {"status": "running", "reports": {}}
        runs[prefix] = entry
        try:
            paths = {}
            for variant in ("v0", "identity", "v16"):
                destination = output / prefix / variant
                settings = {**ANCHOR_CONFIG, "beta_max": 0.0} if variant == "identity" else ANCHOR_CONFIG
                run_dataset(f"{prefix}_{variant}", runner, {
                    **shared, **config, "VARIANT": variant, "OUTPUT": destination,
                    "SMOKE_ONLY": RUN_MODE == "smoke", "ANCHOR_CONFIG": settings,
                }, scripts)
                base = destination / RUN_MODE
                candidates = [base / "run.json"] if prefix == "ucf" else list(base.glob("run_*/run.json"))
                if len(candidates) != 1 or not candidates[0].is_file():
                    raise ValueError(f"Missing report: {prefix}/{variant}")
                paths[variant] = candidates[0]
                entry["reports"][variant] = str(candidates[0].relative_to(output))
            if RUN_MODE == "full":
                rows, fa, entry["diagnostics"] = compare(prefix, paths, analysis)
                metrics.extend(rows)
                alarms.extend(fa)
            else:
                report = json.loads(paths["v16"].read_text())
                # Smoke estimate excludes downloads/input hashing and is only an estimate.
                if prefix == "ucf":
                    seconds, sampled = report["smoke_seconds"], report["video_count"]
                else:
                    seconds, sampled = report["elapsed_seconds"], report["forward_calls"]
                entry["estimated_three_full_passes_seconds"] = (
                    seconds / sampled * (290 if prefix == "ucf" else 800) * 3)
                entry["estimate_note"] = "Rough smoke extrapolation; video lengths vary; excludes I/O/setup"
                print(prefix, "rough full inference seconds:", entry["estimated_three_full_passes_seconds"])
            entry["status"] = "ok"
        except Exception as exc:
            entry.update(status="failed", error=f"{type(exc).__name__}: {exc}")
            print(prefix, entry["error"], flush=True)
        finally:
            (output / "summary.json").write_text(json.dumps({
                **experiment, "runs": runs, "metrics": metrics,
                "elapsed_seconds": time.perf_counter() - started,
            }, indent=2), encoding="utf-8")
    write_csv(output / "comparison.csv", metrics)
    write_csv(output / "false_alarm_diagnostics.csv", alarms)
    for row in metrics:
        print(f"{row['dataset']} {row['branch']} {row['metric']}: "
              f"{row['v0_pct']:.5f}% -> {row['v16_pct']:.5f}% ({row['delta_pp']:+.5f} pp)")
    archive = output / "vadclip_v16_results.zip"
    with ZipFile(archive, "w", ZIP_DEFLATED) as z:
        for path in sorted(output.rglob("*")):
            if path.is_file() and path != archive and "__pycache__" not in path.parts:
                z.write(path, path.relative_to(output))
    with ZipFile(archive) as z:
        if z.testzip() is not None:
            raise ValueError("Output ZIP integrity failed")
    print("Download:", archive)
    if any(r["status"] != "ok" for r in runs.values()):
        raise RuntimeError("Some runs failed. ZIP contains partial reports and logs, not a complete benchmark.")
    return str(archive)


RESULT_ZIP = run_both()
